# NutriLens — Food segmenter (food vs background) · Colab

Trains the segmentation model (`05 §6`): **DeepLabV3-MobileNetV3-Large**, binary
food/not-food, on **FoodSeg103** (already pixel-labelled). No manual labelling.

**One-time setup:** put **`segmenter_train.py`** in Drive at
`MyDrive/nutrilens/segmenter_train.py`.

**Every run:** Runtime → Change runtime type → **T4 GPU**, then *Run all*.
~20–40 min end-to-end.

In [ ]:
# 1. GPU check
!nvidia-smi -L
import torch; assert torch.cuda.is_available(), 'No GPU — set Runtime > T4 GPU'

In [ ]:
# 2. Deps: HuggingFace datasets for FoodSeg103 (torch/torchvision already on Colab).
!pip install -q datasets

In [ ]:
# 3. Mount Drive and pull the training script into this session.
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/nutrilens'
import os, shutil
src = os.path.join(DRIVE, 'segmenter_train.py')
assert os.path.exists(src), f'Upload segmenter_train.py to {DRIVE}/ first'
shutil.copy(src, '/content/segmenter_train.py')
print('Loaded segmenter_train.py from Drive.')

In [ ]:
# 4. Train (DeepLabV3-MobileNetV3, binary food/bg on FoodSeg103). ~20-40 min on a T4.
#    Artifacts go straight to Drive so a disconnect doesn't lose them.
!python /content/segmenter_train.py --train --out-dir /content/drive/MyDrive/nutrilens/segmenter_artifacts

In [ ]:
# 5. Export TorchScript for the backend, then show the IoU.
!python /content/segmenter_train.py --export --out-dir /content/drive/MyDrive/nutrilens/segmenter_artifacts
!cat /content/drive/MyDrive/nutrilens/segmenter_artifacts/metrics.json

In [ ]:
# 6. Download the exported model (drop it into backend/models/food_segmenter.pth).
from google.colab import files
files.download('{DRIVE}/segmenter_artifacts/food_segmenter.pth'.replace('{DRIVE}', '/content/drive/MyDrive/nutrilens'))

### If FoodSeg103 fails to load
The dataset id is `EduardoPacheco/FoodSeg103` (set at the top of
`segmenter_train.py` as `DATASET_ID`). If HuggingFace can't fetch it, swap in
another food-segmentation dataset that exposes an **image** column and an
**integer-mask** column, and re-run — the script auto-detects common column
names and binarises any mask id > 0 to "food".